In [5]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import os
import random

# Load UMLS vocabulary

In [8]:
umls = pd.read_csv("./data/umls/mrconso_filtered_db_and_sty_20251116.csv")
umls['cui'] = umls['cui'].str.strip()


In [10]:
umls.shape, umls['cui'].nunique()

((2566153, 21), 902598)

In [4]:
umls.head()

,cui,lat,ts,lui,stt,sui,ispref,aui,saui,scui,...,sab,tty,code,str,srl,suppress,cvf,dummy,tui,sty
0,C5686927,ENG,P,L18212371,PF,S21873009,Y,A34734806,4.954920e+09,1204128009,...,SNOMEDCT_US,PT,1204128009,Cholesterol atheroembolism of cerebrum,9,N,256.0,,T047,Disease or Syndrome
1,C5686927,ENG,P,L18212371,VO,S21873011,Y,A34735861,4.952425e+09,1204128009,...,SNOMEDCT_US,SY,1204128009,Cholesterol atheroembolism to cerebrum,9,N,256.0,,T047,Disease or Syndrome
2,C5686927,ENG,S,L18199483,PF,S21873014,Y,A34750373,4.954921e+09,1204128009,...,SNOMEDCT_US,SY,1204128009,Cholesterol atheromatous embolism of cerebrum,9,N,256.0,,T047,Disease or Syndrome
3,C5686927,ENG,S,L18205958,PF,S21873013,Y,A34721536,4.954922e+09,1204128009,...,SNOMEDCT_US,FN,1204128009,Cholesterol atheromatous embolism of cerebrum ...,9,N,NaN,,T047,Disease or Syndrome
4,C5686991,ENG,P,L18212257,PF,S21872034,Y,A34719839,4.955543e+09,15703881000119104,...,SNOMEDCT_US,PT,15703881000119104,Bilateral contracture of muscle of thighs,9,N,256.0,,T190,Anatomical Abnormality


In [5]:
umls.loc[umls['cui'] == 'C5940806']

,cui,lat,ts,lui,stt,sui,ispref,aui,saui,scui,...,sab,tty,code,str,srl,suppress,cvf,dummy,tui,sty
1165751,C5940806,ENG,P,L19575909,PF,S23359038,Y,A36799650,NaN,M000768046,...,MSH,PEP,D011559,Secondary Intracranial Hypertension,0,N,NaN,,T047,Disease or Syndrome


In [6]:
umls.groupby(['sab', 'srl']).size().reset_index(name='row_count').sort_values(by='row_count', ascending=False)

,sab,srl,row_count
6,MSH,0,1157213
10,SNOMEDCT_US,9,693711
9,RXNORM,0,198839
3,ICD10CM,4,110943
5,MDR,3,103188
8,OMIM,0,88455
7,NDDF,3,63024
0,DRUGBANK,0,44646
1,HPO,0,41571
11,VANDF,0,31314


In [7]:
umls.groupby('sty').size().reset_index(name='row_count').sort_values(by='row_count', ascending=False)

,sty,row_count
33,Organic Chemical,416230
1,"Amino Acid, Peptide, or Protein",389172
15,Disease or Syndrome,292937
13,Clinical Drug,252778
20,Finding,195000
6,Biologically Active Substance,190663
26,Injury or Poisoning,131547
39,Therapeutic or Preventive Procedure,115313
18,Enzyme,94756
31,Neoplastic Process,66443


#### filter UMLS

In [8]:
umls.shape

(2566153, 21)

In [9]:
umls = umls[umls['srl'].isin([0, 9])]
umls.shape

(2276727, 21)

In [10]:
# Step 1: Filter rows with ispref = 'Y' or tty = 'PT'
preferred_rows = umls[(umls['ispref'] == 'Y') | (umls['tty'] == 'PT')]

# Step 2: Identify CUIs not covered in the filtered set
remaining_cuis = set(umls['cui']) - set(preferred_rows['cui'])

# Step 3: Select any row for the remaining CUIs
fallback_rows = umls[umls['cui'].isin(remaining_cuis)].drop_duplicates(subset='cui', keep='first')

# Step 4: Combine both sets of rows
unique_cui_umls = pd.concat([preferred_rows, fallback_rows]).drop_duplicates(subset='cui', keep='first')


In [11]:
unique_cui_umls.shape, unique_cui_umls['cui'].nunique()

((768662, 21), 768662)

In [12]:
unique_cui_umls.head()

,cui,lat,ts,lui,stt,sui,ispref,aui,saui,scui,...,sab,tty,code,str,srl,suppress,cvf,dummy,tui,sty
0,C5686927,ENG,P,L18212371,PF,S21873009,Y,A34734806,4.954920e+09,1204128009,...,SNOMEDCT_US,PT,1204128009,Cholesterol atheroembolism of cerebrum,9,N,256.0,,T047,Disease or Syndrome
4,C5686991,ENG,P,L18212257,PF,S21872034,Y,A34719839,4.955543e+09,15703881000119104,...,SNOMEDCT_US,PT,15703881000119104,Bilateral contracture of muscle of thighs,9,N,256.0,,T190,Anatomical Abnormality
8,C5687082,ENG,P,L18199014,PF,S21888708,Y,A34740358,4.964107e+09,323051000119106,...,SNOMEDCT_US,PT,323051000119106,Unequal lower limb length due to acquired shor...,9,N,256.0,,T020,Acquired Abnormality
10,C5687103,ENG,P,L18203626,PF,S21871656,Y,A34742044,4.964333e+09,1208415001,...,SNOMEDCT_US,PT,1208415001,Autosomal dominant congenital fiber-type dispr...,9,N,256.0,,T047,Disease or Syndrome
15,C5687149,ENG,P,L18213926,PF,S21887991,Y,A34723764,4.964213e+09,322291000119100,...,SNOMEDCT_US,PT,322291000119100,Synovial hypertrophy of left hand,9,N,256.0,,T046,Pathologic Function


In [13]:
#C0066677)
unique_cui_umls[unique_cui_umls['cui']=='C0066677']

,cui,lat,ts,lui,stt,sui,ispref,aui,saui,scui,...,sab,tty,code,str,srl,suppress,cvf,dummy,tui,sty
2249814,C0066677,ENG,P,L0066677,PF,S0178994,Y,A10337535,438965.0,30125,...,RXNORM,IN,30125,modafinil,0,N,4352.0,,T109,Organic Chemical


In [14]:
unique_cui_umls.groupby('srl').size().reset_index(name='row_count').sort_values(by='row_count', ascending=False)

,srl,row_count
0,0,560390
1,9,208272


In [15]:
unique_cui_umls.groupby('sab').size().reset_index(name='row_count').sort_values(by='row_count', ascending=False)

,sab,row_count
3,MSH,364882
6,SNOMEDCT_US,208272
5,RXNORM,112150
4,OMIM,41651
1,HPO,16874
2,ICD9CM,13695
0,DRUGBANK,6374
7,VANDF,4764


In [16]:
unique_cui_umls.groupby('sty').size().reset_index(name='row_count').sort_values(by='row_count', ascending=False)

,sty,row_count
33,Organic Chemical,199684
13,Clinical Drug,98293
20,Finding,83328
6,Biologically Active Substance,72107
15,Disease or Syndrome,58215
39,Therapeutic or Preventive Procedure,38842
1,"Amino Acid, Peptide, or Protein",30375
18,Enzyme,30083
4,Bacterium,26025
26,Injury or Poisoning,23659


In [17]:
cui_sty_dict = unique_cui_umls.set_index("cui")["sty"].to_dict()
cui_sty_dict['C0066677']

'Organic Chemical'

In [ ]:
# List of drug/chemical-related STYs (from your previous filtered list)
drug_chemical_stys = [
    "Organic Chemical",
    "Clinical Drug",
    "Biologically Active Substance",
    "Amino Acid, Peptide, or Protein",
    "Enzyme",
    "Immunologic Factor",
    "Nucleic Acid, Nucleoside, or Nucleotide",
    "Inorganic Chemical",
    "Antibiotic",
    "Biomedical or Dental Material",
    "Hormone",
    "Element, Ion, or Isotope",
    "Drug Delivery Device",
    "Vitamin",
    "Chemical Viewed Structurally",
    "Chemical"
]

# Filter rows where 'sty' is in the drug/chemical list
filtered_df = unique_cui_umls[unique_cui_umls['sty'].isin(drug_chemical_stys)]
print(filtered_df.shape)

#filtered_df.to_csv(f"data/umls/mrconso_filtered_db_and_sty_{len(filtered_df)}_drug_chemical_level_0_9.csv", index=False)

In [18]:
unique_cui_umls[['cui','srl','sab','str', 'sty']].to_csv(f"data/umls/mrconso_filtered_db_and_sty_{len(unique_cui_umls)}_level_0_9.csv", index=False)

In [32]:
# partition function
def partition(arr, low, high):
    
    # choose the pivot
    pivot = arr[high]
    
    # index of smaller element and indicates 
    # the right position of pivot found so far
    i = low - 1
    
    # traverse arr[low..high] and move all smaller
    # elements to the left side. Elements from low to 
    # i are smaller after every iteration
    for j in range(low, high):
        if arr[j] < pivot:
            i += 1
            swap(arr, i, j)
    
    # move pivot after smaller elements and
    # return its position
    swap(arr, i + 1, high)
    return i + 1

# swap function
def swap(arr, i, j):
    arr[i], arr[j] = arr[j], arr[i]

# the QuickSort function implementation
def quickSort(arr, low, high):
    if low < high:
        print(arr, low, high)
        # pi is the partition return index of pivot
        pi = partition(arr, low, high)
        print(pi)
        # recursion calls for smaller elements
        # and greater or equals elements
        quickSort(arr, low, pi - 1)
        quickSort(arr, pi + 1, high)

In [34]:
arr = [5, 2, 3, 8, 4, 7, 1]
n = len(arr)

quickSort(arr, 0, n - 1)

for val in arr:
    print(val, end=" ")

[5, 2, 3, 8, 4, 7, 1] 0 6
0
[1, 2, 3, 8, 4, 7, 5] 1 6
4
[1, 2, 3, 4, 5, 7, 8] 1 3
3
[1, 2, 3, 4, 5, 7, 8] 1 2
2
[1, 2, 3, 4, 5, 7, 8] 5 6
6
1 2 3 4 5 7 8 